In [ ]:
#영상에서 객체탐지와 위험객체 표시하기 

In [1]:
import cv2
from ultralytics import YOLO


In [2]:
import urllib.request
url = 'https://github.com/isl-org/MiDaS/releases/download/v2_1/model-small.onnx'
save_path = 'midas_small.onnx'
urllib.request.urlretrieve(url,save_path)
print("다운로드")

다운로드


In [3]:
midas_model= cv2.dnn.readNet("midas_small.onnx")
frame_idx=0
depth_map=None

In [9]:
def estimate_depth(frame):
    blob = cv2.dnn.blobFromImage(frame,1/255.0,(256,256),swapRB=True,crop=False)
    midas_model.setInput(blob)
    output=midas_model.forward()
    depth_map=output[0,:,:]
    return cv2.resize(
        cv2.normalize(depth_map, None, 0,1,cv2.NORM_MINMAX),
        (frame.shape[1],frame.shape[0])
    )

In [5]:
#위험한 객체를 추가해준다.
DANGER_CLASSES=['car','bus','truck','person','bicycle','traffic light']

In [6]:
cap = cv2.VideoCapture('walk.mp4')

In [7]:
model = YOLO('yolov8n.pt')


In [ ]:
if cap.isOpened()==False:
    print("Failed to open video")
    exit()
while True:
    ret,frame = cap.read()
    if frame_idx % 5 == 0 or depth_map is None:
        
        depth_map = estimate_depth(frame)
    if ret == False:
        break
    results = model(frame, verbose=False)[0]
    frame= results.plot()
    
    for box in results.boxes:
        cls_id = int(box.cls[0])
        label =model.names[cls_id]
        if label in DANGER_CLASSES:
            X1,Y1,X2,Y2 = map(int, box.xyxy[0])
            cx,cy= int((X1+X2)/2),int((Y1+Y2)/2)
            depth_score=depth_map[cy,cx]
            if depth_score>0.6:
                
                cv2.putText(frame,f"{label} detected!",(X1,Y1 -10),
                           cv2.FONT_HERSHEY_SIMPLEX,0.8,(0,0,255),2)
    
    cv2.imshow("GuidePath",frame)
    key=cv2.waitKey(50) & 0xFF
    if key == ord('q'):
        break
    frame_idx+=1
cap.release()
cv2.destroyAllWindows()